# EDA - `player_appearance_run.csv`

Lean EDA of the run / sprint event table. Two deliverables:

1. **Data cleaning** - structural integrity, dtype coercion, bounds, orphans.
2. **Feature manifest** - which existing main-table run columns and which
   engineered candidates carry signal for `scored_after`.

Section structure:

| ID | Section | Purpose |
|----|---------|---------|
| A | Data cleaning | structural + value-domain checks |
| B | Replicate main-table aggregates | confirm windowing rule |
| C | Bivariate signal (main run cols) | triage existing features |
| D | Engineered candidates | propose + evaluate new features |
| E | Redundancy + significance | drop collinear / non-significant |
| F | Feature manifest | final keep / drop list |


## Setup

In [ ]:
"""Imports."""
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd


In [13]:
"""Display options."""
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 60)
pd.set_option("display.precision", 4)


In [14]:
"""Paths."""
PROJECT_ROOT: Path = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
DATA_DIR: Path = PROJECT_ROOT / "data"
RUN_PATH: Path = DATA_DIR / "player_appearance_run.csv"
MAIN_PATH: Path = DATA_DIR / "players_quarters_final.csv"
assert RUN_PATH.exists() and MAIN_PATH.exists()


In [15]:
"""Load tables.

`min_speed`, `max_speed`, `distance` arrive as quoted strings in the source
CSV; cast explicitly so all later numeric ops work. We keep the raw frame
unmodified beyond the cast.
"""
runs: pd.DataFrame = pd.read_csv(RUN_PATH)
for col in ("min_speed", "max_speed", "distance"):
    runs[col] = pd.to_numeric(runs[col], errors="coerce")

main: pd.DataFrame = pd.read_csv(MAIN_PATH)

appearance_meta: pd.DataFrame = (
    main[["player_appearance_id", "player_id", "fixture_id", "position",
          "is_home", "minute_in", "minute_out"]]
    .drop_duplicates(subset="player_appearance_id")
    .set_index("player_appearance_id")
)
runs.shape, main.shape, appearance_meta.shape


((35133, 10), (3486, 33), (869, 6))

In [ ]:
#!pip install scipy

  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
"""Helpers (Wilson CI + binned-rate table)."""
from scipy import stats as _stats


def wilson_ci(successes: int, trials: int, alpha: float = 0.05) -> tuple[float, float]:
    if trials == 0:
        return float("nan"), float("nan")
    p = successes / trials
    z = _stats.norm.ppf(1.0 - alpha / 2.0)
    denom = 1.0 + (z ** 2) / trials
    centre = (p + (z ** 2) / (2.0 * trials)) / denom
    half = z * np.sqrt(p * (1.0 - p) / trials + (z ** 2) / (4.0 * trials ** 2)) / denom
    return float(centre - half), float(centre + half)


def binned_rate(df: pd.DataFrame, feature: str, target: str = "scored_after",
                bins=None, by: str | None = None) -> pd.DataFrame:
    work = df[[feature, target] + ([by] if by else [])].copy()
    if bins is not None:
        work["__bin"] = pd.cut(work[feature], bins=bins, include_lowest=True)
    else:
        work["__bin"] = work[feature]
    grouping = ["__bin"] if by is None else [by, "__bin"]
    g = work.groupby(grouping, observed=False, dropna=False)[target].agg(["size", "sum"])
    g = g.rename(columns={"size": "n", "sum": "n_pos"}).reset_index()
    g["rate"] = g["n_pos"] / g["n"].where(g["n"] > 0)
    ci = g.apply(lambda r: wilson_ci(int(r["n_pos"]), int(r["n"])), axis=1, result_type="expand")
    g["ci_lo"], g["ci_hi"] = ci[0], ci[1]
    return g.rename(columns={"__bin": "bin"})


PERIOD_ORDER = {"half_1": 0, "half_2": 1, "extra_time_1": 2, "extra_time_2": 3}


## Section A - Data cleaning

Six compact checks. Each cell prints its own verdict; a roll-up table at the
end of the section captures the decisions carried into Section B.

| ID | Check |
|----|-------|
| A1 | Shape, dtypes, head |
| A2 | Primary-key uniqueness on `id` |
| A3 | Missingness on every column |
| A4 | Domain checks on `period`, `run_type`, `stage` |
| A5 | Numeric bounds (min<=max speed, positive distance, plausible m/s) |
| A6 | Stoppage-time encoding (`minute > period_cap`) |
| A7 | Orphan appearances (run rows whose `player_appearance_id` is absent from `main`) |


### A1 - Shape & dtypes

In [21]:
print("shape:", runs.shape)
print()
print(runs.dtypes.to_string())
runs.head(3)


shape: (35133, 10)

id                        int64
period                   object
stage                    object
possession                int64
run_type                 object
minute                    int64
min_speed               float64
max_speed               float64
distance                float64
player_appearance_id      int64


,id,period,stage,possession,run_type,minute,min_speed,max_speed,distance,player_appearance_id
0,703325,half_1,top,2345,hsr,26,5.52,5.71,4.9674,40764
1,703332,half_1,bottom,2345,hsr,37,5.56,6.99,7.1286,40764
2,703314,half_1,middle,2345,hsr,5,4.07,6.90,34.6272,40764


### A2 - PK uniqueness on `id`

In [22]:
dupes = runs["id"].duplicated().sum()
print(f"duplicate `id` rows: {dupes}")
print(f"unique ids: {runs['id'].nunique()} / {len(runs)}")


duplicate `id` rows: 0
unique ids: 35133 / 35133


### A3 - Missingness

In [23]:
na = runs.isna().sum()
na = na[na > 0]
print("columns with NaN:" if len(na) else "no NaN anywhere")
print(na.to_string() if len(na) else "")


no NaN anywhere



### A4 - Domain checks

In [24]:
for c in ("period", "run_type", "stage"):
    print(f"{c}: {sorted(runs[c].dropna().unique().tolist())}")
print(f"period values per row count = {runs['period'].value_counts().to_dict()}")
print(f"run_type values per row count = {runs['run_type'].value_counts().to_dict()}")
print(f"stage values per row count = {runs['stage'].value_counts().to_dict()}")


period: ['extra_time_1', 'extra_time_2', 'half_1', 'half_2']
run_type: ['hsr', 'sprint']
stage: ['bottom', 'middle', 'top']
period values per row count = {'half_2': 17519, 'half_1': 17045, 'extra_time_1': 295, 'extra_time_2': 274}
run_type values per row count = {'hsr': 28417, 'sprint': 6716}
stage values per row count = {'middle': 14874, 'top': 10671, 'bottom': 9588}


### A5 - Numeric bounds

Three invariants:

* `min_speed <= max_speed` (must hold by definition of the columns).
* `distance > 0` (a run with zero distance is not a run).
* Speed plausibility: world-record sprint peak is ~12 m/s. Any run with
  `max_speed > 12` is suspicious; any with `min_speed < 4` would mean the
  run never reached the high-speed-running threshold.


In [25]:
bad_order = (runs["min_speed"] > runs["max_speed"]).sum()
zero_dist = (runs["distance"] <= 0).sum()
implausible_high = (runs["max_speed"] > 12).sum()
implausible_low = (runs["min_speed"] < 4).sum()
print(f"min > max: {bad_order}")
print(f"distance <= 0: {zero_dist}")
print(f"max_speed > 12 m/s: {implausible_high}")
print(f"min_speed < 4 m/s: {implausible_low}")
print()
print("speed describe:")
print(runs[["min_speed", "max_speed", "distance"]].describe().to_string())


min > max: 0
distance <= 0: 0
max_speed > 12 m/s: 10
min_speed < 4 m/s: 79

speed describe:
        min_speed   max_speed    distance
count  35133.0000  35133.0000  35133.0000
mean       5.3516      6.7171     47.9274
std        0.8952      0.7981    121.7279
min        0.2900      5.4400      2.8923
25%        5.3900      6.0600      7.7195
50%        5.5100      6.7200     12.0748
75%        5.5400      7.0000     19.9546
max        7.2800     14.1400   1694.0080


### A6 - Stoppage-time encoding

The data dictionary defines period caps at 45 (halves) and 15 (extra time).
Any `minute > cap` is stoppage time. We quantify how many rows fall there
because (i) the strict windowing rule used in shots EDA F2 will assign these
to the `_45` / `ET1_15` checkpoints only via the `last15` open interval and
(ii) the cumulative count must include them at later checkpoints.


In [26]:
cap = {"half_1": 45, "half_2": 45, "extra_time_1": 15, "extra_time_2": 15}
runs["__cap"] = runs["period"].map(cap)
stoppage = runs[runs["minute"] > runs["__cap"]]
print(f"stoppage-time runs: {len(stoppage)} ({len(stoppage)/len(runs):.2%})")
print()
print("by period:")
print(stoppage.groupby("period")["minute"].agg(["count", "min", "max"]).to_string())
runs.drop(columns="__cap", inplace=True)


stoppage-time runs: 2492 (7.09%)

by period:
              count  min  max
period                       
extra_time_1      6   16   16
extra_time_2     43   16   19
half_1          752   46   50
half_2         1691   46   59


### A7 - Orphan appearances

Run events whose `player_appearance_id` is not in the main panel cannot be
attached to any prediction row. We enumerate them once so all later
aggregations filter consistently.


In [27]:
main_apps = set(main["player_appearance_id"].unique())
run_apps = set(runs["player_appearance_id"].unique())
orphans = run_apps - main_apps
print(f"distinct appearances in runs: {len(run_apps)}")
print(f"distinct appearances in main: {len(main_apps)}")
print(f"orphan appearances (in runs but not main): {len(orphans)}")
print(f"run rows attached to orphans: {runs['player_appearance_id'].isin(orphans).sum()}")


distinct appearances in runs: 968
distinct appearances in main: 869
orphan appearances (in runs but not main): 108
run rows attached to orphans: 1010


### Section A - findings

Each row below is filled in after the checks above run.

| Check | Expected | Result |
|-------|----------|--------|
| A1 | shape ~(35133, 10), correct dtypes after cast | filled at runtime |
| A2 | every `id` unique | filled at runtime |
| A3 | no NaN (or only the 3 cast columns) | filled at runtime |
| A4 | period in 4 labels, run_type in {hsr, sprint}, stage in 3 zones | filled at runtime |
| A5 | min<=max, distance>0, max<=12 m/s | filled at runtime |
| A6 | <10% stoppage; H1 max~48, H2 max~57 | filled at runtime |
| A7 | a small handful of orphan appearances | filled at runtime |

**Decisions logged for Section B:**

* Filter run events to `player_appearance_id in main` before any aggregation.
* Cast `min_speed`, `max_speed`, `distance` to numeric on load (already done).
* Use the strict windowing rule from shots EDA F2 (carried over).


## Section B - Replicate the 5 existing main-table aggregates

For each of the five run-derived main columns we recompute the value from raw
events using the strict windowing rule and report the per-row match rate
against the main panel. Columns:

| Main column | Formula on raw runs |
|-------------|---------------------|
| `*_sprints` | count where `run_type == "sprint"` |
| `*_hsr`     | count where `run_type == "hsr"`    |
| `*_distance` | sum of `distance` over all runs in window |
| `*_mean_max_speed` | mean of `max_speed` over all runs in window |
| `*_peak_speed` | max of `max_speed` over all runs in window |

`*_` is `last15` or `cumul`. Strict rules carry from shots EDA F2:

* **cumul**: `(period_order, minute) <= (cp_period_order, cp_minute)`.
* **last15**: same period AND `cp_minute - 15 < minute <= cp_minute`.

A near-perfect (>99%) match rate confirms we can trust the contract and reuse
it both for engineered features and for any `last15`/`cumul` aggregations
beyond the five provided columns.


In [28]:
"""Build a clean run frame keyed for windowing."""
runs_clean = runs[runs["player_appearance_id"].isin(main_apps)].copy()
runs_clean["period_order"] = runs_clean["period"].map(PERIOD_ORDER)
print(f"runs_clean shape: {runs_clean.shape}")


runs_clean shape: (34123, 11)


In [29]:
"""Replicate aggregates per (player_appearance_id, checkpoint).

We loop over checkpoints (only 7) and use vectorised filters per checkpoint;
for 35k rows x 7 checkpoints this completes in well under a second.
"""
CHECKPOINTS = [
    ("H1_15", "half_1", 15), ("H1_30", "half_1", 30), ("H1_45", "half_1", 45),
    ("H2_15", "half_2", 15), ("H2_30", "half_2", 30), ("H2_45", "half_2", 45),
    ("ET1_15", "extra_time_1", 15),
]
CP_ORDER = {cp: PERIOD_ORDER[per] for cp, per, _ in CHECKPOINTS}
CP_MINUTE = {cp: m for cp, _, m in CHECKPOINTS}


def _agg(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("player_appearance_id")
    out = pd.DataFrame({
        "sprints": g.apply(lambda x: (x["run_type"] == "sprint").sum()),
        "hsr": g.apply(lambda x: (x["run_type"] == "hsr").sum()),
        "distance": g["distance"].sum(),
        "mean_max_speed": g["max_speed"].mean(),
        "peak_speed": g["max_speed"].max(),
    })
    return out


def replicate_for_cp(cp: str) -> pd.DataFrame:
    cp_per = next(p for c, p, _ in CHECKPOINTS if c == cp)
    cp_min = CP_MINUTE[cp]
    cp_ord = CP_ORDER[cp]

    cumul_mask = (
        (runs_clean["period_order"] < cp_ord) |
        ((runs_clean["period_order"] == cp_ord) & (runs_clean["minute"] <= cp_min))
    )
    last15_mask = (
        (runs_clean["period"] == cp_per) &
        (runs_clean["minute"] > cp_min - 15) &
        (runs_clean["minute"] <= cp_min)
    )

    cumul = _agg(runs_clean[cumul_mask]).add_prefix("cumul_repl_")
    last15 = _agg(runs_clean[last15_mask]).add_prefix("last15_repl_")
    out = cumul.join(last15, how="outer").fillna(0)
    out["checkpoint"] = cp
    return out.reset_index()


repl = pd.concat([replicate_for_cp(cp) for cp, _, _ in CHECKPOINTS], ignore_index=True)
repl.shape


C:\Users\tymot\AppData\Local\Temp\ipykernel_31352\345800139.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "sprints": g.apply(lambda x: (x["run_type"] == "sprint").sum()),
C:\Users\tymot\AppData\Local\Temp\ipykernel_31352\345800139.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  "hsr": g.apply(lambda x: (x["run_type"] == "hsr").sum()),
C:\Users\tymot\AppData\Local\Temp\ipykernel_31352\345800139.p

(5227, 12)

In [ ]:
"""Compare replicated values against main."""
join_cols = ["player_appearance_id", "checkpoint"]
target_cols = [
    ("cumul_sprints", "cumul_repl_sprints"),
    ("cumul_hsr", "cumul_repl_hsr"),
    ("cumul_distance", "cumul_repl_distance"),
    ("cumul_mean_max_speed", "cumul_repl_mean_max_speed"),
    ("cumul_peak_speed", "cumul_repl_peak_speed"),
    ("last15_sprints", "last15_repl_sprints"),
    ("last15_hsr", "last15_repl_hsr"),
    ("last15_distance", "last15_repl_distance"),
    ("last15_mean_max_speed", "last15_repl_mean_max_speed"),
    ("last15_peak_speed", "last15_repl_peak_speed"),
]

merged = main[join_cols + [m for m, _ in target_cols]].merge(repl, on=join_cols, how="left").fillna(0)


def cmp(a: pd.Series, b: pd.Series, tol: float = 1e-3) -> tuple[float, int]:
    diff = (a - b).abs()
    match = (diff <= tol * (1.0 + a.abs())) | (diff <= tol)
    return float(match.mean()), int((~match).sum())


rows = []
for main_col, repl_col in target_cols:
    rate, n_bad = cmp(merged[main_col], merged[repl_col])
    rows.append({"feature": main_col, "match_rate": rate, "n_mismatch": n_bad})

cmp_df = pd.DataFrame(rows)
cmp_df


,feature,match_rate,n_mismatch
0,cumul_sprints,1.0000,0
1,cumul_hsr,1.0000,0
2,cumul_distance,1.0000,0
3,cumul_mean_max_speed,1.0000,0
4,cumul_peak_speed,1.0000,0
5,last15_sprints,0.9644,124
6,last15_hsr,0.8987,353
7,last15_distance,0.8959,363
8,last15_mean_max_speed,0.9010,345
9,last15_peak_speed,0.9779,77


: 

### Section B - findings

Read off the table above. Targets:

* **counts (sprints, hsr)** should match exactly (>99.9%); any gap indicates
  a windowing-rule disagreement and must be investigated before trusting
  the engineered features.
* **distance / mean_max_speed / peak_speed** can have small rounding
  residuals (floating-point sums); >99% match rate at 1e-3 relative
  tolerance is acceptable.
* If `last15` rates fall noticeably below `cumul` rates, the discrepancy is
  the same stoppage-time slack we saw in shots EDA F2 (period-cap
  checkpoints).

If everything matches we adopt the strict rule unchanged for Section D
engineered features. If mismatches are large for one specific column we
log it and either patch the rule or downgrade the dependent feature.


## Section C - Bivariate signal: existing main-table run columns

Triage the 10 existing run-derived main-table columns. For each we report
the binned `scored_after` rate with a Wilson 95% CI, both pooled and
stratified by position. Anything flat after stratification is a drop
candidate.

The bin edges are chosen to put roughly the same mass in each bucket and
to make the zero bucket explicit (a large fraction of rows for short
appearances and early checkpoints carry zeros for the cumulative columns).


In [16]:
"""Bin definitions per feature.

We use coarse bins (3-4 buckets) to stay above the n>=30 rule of thumb at
the strongest stratum. `[-inf, 0]` isolates the zero bucket cleanly.
"""
RUN_BIN_SPECS = {
    "last15_sprints":        [-1, 0, 1, 3, 100],
    "last15_hsr":            [-1, 0, 2, 5, 100],
    "last15_distance":       [-1, 0, 30, 80, 1e6],
    "last15_mean_max_speed": [-1, 0, 5.5, 6.5, 100],
    "last15_peak_speed":     [-1, 0, 6, 7, 100],
    "cumul_sprints":         [-1, 0, 3, 7, 200],
    "cumul_hsr":             [-1, 0, 5, 12, 200],
    "cumul_distance":        [-1, 0, 100, 250, 1e6],
    "cumul_mean_max_speed":  [-1, 0, 5.5, 6.5, 100],
    "cumul_peak_speed":      [-1, 0, 6, 7, 100],
}
BASELINE = main["scored_after"].mean()
print(f"baseline scored_after rate = {BASELINE:.4f}")


baseline scored_after rate = 0.0582


In [17]:
"""Pooled bivariate rates."""
pooled = []
for feat, edges in RUN_BIN_SPECS.items():
    t = binned_rate(main, feat, "scored_after", bins=edges)
    t["feature"] = feat
    pooled.append(t)
pooled_df = pd.concat(pooled, ignore_index=True)[["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]]
pooled_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,last15_sprints,"(-1.001, 0.0]",1033,40,0.0387,2.8564e-02,0.0523
1,last15_sprints,"(0.0, 1.0]",984,70,0.0711,5.6689e-02,0.0889
2,last15_sprints,"(1.0, 3.0]",1127,74,0.0657,5.2625e-02,0.0816
3,last15_sprints,"(3.0, 100.0]",342,19,0.0556,3.5851e-02,0.0851
4,last15_hsr,"(-1.001, 0.0]",254,0,0.0000,0.0000e+00,0.0149
5,last15_hsr,"(0.0, 2.0]",323,12,0.0372,2.1378e-02,0.0638
6,last15_hsr,"(2.0, 5.0]",861,58,0.0674,5.2470e-02,0.0861
7,last15_hsr,"(5.0, 100.0]",2048,133,0.0649,5.5062e-02,0.0764
8,last15_distance,"(-1.001, 0.0]",243,0,0.0000,0.0000e+00,0.0156
9,last15_distance,"(0.0, 30.0]",242,7,0.0289,1.4081e-02,0.0585


In [18]:
"""Stratified by position (drop G - target identically zero)."""
strat = []
sub = main[main["position"].isin(["A", "M", "D"])].copy()
for feat, edges in RUN_BIN_SPECS.items():
    t = binned_rate(sub, feat, "scored_after", bins=edges, by="position")
    t["feature"] = feat
    strat.append(t)
strat_df = pd.concat(strat, ignore_index=True)[["feature", "position", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]]
strat_df


,feature,position,bin,n,n_pos,rate,ci_lo,ci_hi
0,last15_sprints,A,"(-1.001, 0.0]",95,16,0.1684,0.1064,0.2562
1,last15_sprints,A,"(0.0, 1.0]",160,24,0.1500,0.1029,0.2135
2,last15_sprints,A,"(1.0, 3.0]",249,30,0.1205,0.0857,0.1668
3,last15_sprints,A,"(3.0, 100.0]",111,7,0.0631,0.0309,0.1245
4,last15_sprints,D,"(-1.001, 0.0]",344,11,0.0320,0.0179,0.0563
...,...,...,...,...,...,...,...,...
115,cumul_peak_speed,D,"(7.0, 100.0]",1168,32,0.0274,0.0195,0.0384
116,cumul_peak_speed,M,"(-1.001, 0.0]",2,0,0.0000,0.0000,0.6576
117,cumul_peak_speed,M,"(0.0, 6.0]",2,0,0.0000,0.0000,0.6576
118,cumul_peak_speed,M,"(6.0, 7.0]",67,3,0.0448,0.0153,0.1236


### Section C - findings

Read off the two tables above. We expect to see (analogous to the shots
EDA findings):

* `last15_*` showing cleaner monotone signal than `cumul_*` (recency wins
  for shots; check if it also wins for runs).
* Speed columns (`mean_max_speed`, `peak_speed`) likely flat - speed alone
  doesn't predict scoring, only volume / distance does.
* Sprints likely the single strongest run column at the bin level.

The verdict per feature feeds the manifest in Section F.


## Section D - Engineered candidates

We propose 8 candidate features derived from the raw run table. Each is
defined at the (player_appearance_id, checkpoint) level using the strict
windowing rule from Section B.

| # | Candidate | Formula | Rationale |
|---|-----------|---------|-----------|
| 1 | `sprint_share`            | `cumul_sprints / (cumul_sprints + cumul_hsr)` | mix of explosiveness vs sub-maximal effort |
| 2 | `runs_per_minute_played`  | `(cumul_sprints + cumul_hsr) / minutes_on_pitch_so_far` | load intensity normalised by exposure |
| 3 | `top_third_run_share`     | share of cumul runs with `stage == "top"` | tactical proxy for attacking presence |
| 4 | `peak_minus_mean_speed`   | `cumul_peak_speed - cumul_mean_max_speed` | variability / explosiveness gap |
| 5 | `distance_per_run`        | `cumul_distance / (cumul_sprints + cumul_hsr)` | average burst length |
| 6 | `last15_sprint_uplift`    | `last15_sprints / 15  /  (cumul_sprints / minutes_so_far + eps)` | RQ6 anchor (recency vs baseline) |
| 7 | `last15_distance_uplift`  | analogous on distance | RQ6 anchor (volume version) |
| 8 | `top_third_last15_share`  | share of `last15` runs with `stage == "top"` | recent attacking-third presence |

Each candidate is added to a panel keyed on `(player_appearance_id,
checkpoint)` and merged onto `main` for bivariate evaluation against
`scored_after`.


In [19]:
"""Build engineered features per (appearance, checkpoint)."""
EPS = 1e-6


def aggregate_with_stages(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("player_appearance_id")
    out = pd.DataFrame({
        "n_runs": g.size(),
        "sprints": g.apply(lambda x: (x["run_type"] == "sprint").sum()),
        "hsr": g.apply(lambda x: (x["run_type"] == "hsr").sum()),
        "distance": g["distance"].sum(),
        "mean_max_speed": g["max_speed"].mean(),
        "peak_speed": g["max_speed"].max(),
        "n_top_runs": g.apply(lambda x: (x["stage"] == "top").sum()),
    })
    return out


def features_for_cp(cp: str) -> pd.DataFrame:
    cp_per = next(p for c, p, _ in CHECKPOINTS if c == cp)
    cp_min = CP_MINUTE[cp]
    cp_ord = CP_ORDER[cp]

    cumul_mask = (
        (runs_clean["period_order"] < cp_ord) |
        ((runs_clean["period_order"] == cp_ord) & (runs_clean["minute"] <= cp_min))
    )
    last15_mask = (
        (runs_clean["period"] == cp_per) &
        (runs_clean["minute"] > cp_min - 15) &
        (runs_clean["minute"] <= cp_min)
    )

    cumul = aggregate_with_stages(runs_clean[cumul_mask]).add_prefix("cumul_")
    last15 = aggregate_with_stages(runs_clean[last15_mask]).add_prefix("last15_")
    out = cumul.join(last15, how="outer").fillna(0)
    out["checkpoint"] = cp
    return out.reset_index()


feat_panel = pd.concat([features_for_cp(cp) for cp, _, _ in CHECKPOINTS], ignore_index=True)
feat_panel.shape


(5227, 16)

In [20]:
"""Compute the 8 engineered features."""
# Match-minute-at-cp -> minutes "played so far" assuming player was on the
# pitch the whole window we are looking at. We approximate by the full
# elapsed match minute (consistent with main-panel aggregates).
def match_minute(cp: str) -> int:
    return {
        "H1_15": 15, "H1_30": 30, "H1_45": 45,
        "H2_15": 60, "H2_30": 75, "H2_45": 90, "ET1_15": 105,
    }[cp]


feat_panel["match_minute_at_cp"] = feat_panel["checkpoint"].map(match_minute)

# Fold in minute_in / minute_out so we can compute a player-specific
# minutes-played-so-far. minute_in/out are continuous match minutes.
am = appearance_meta.reset_index()[["player_appearance_id", "minute_in", "minute_out"]]
feat_panel = feat_panel.merge(am, on="player_appearance_id", how="left")
feat_panel["mins_played_so_far"] = (
    feat_panel[["match_minute_at_cp", "minute_out"]].min(axis=1)
    - feat_panel["minute_in"].clip(lower=1) + 1
).clip(lower=1)

feat = pd.DataFrame(index=feat_panel.index)
feat["player_appearance_id"] = feat_panel["player_appearance_id"]
feat["checkpoint"] = feat_panel["checkpoint"]

cumul_runs = feat_panel["cumul_sprints"] + feat_panel["cumul_hsr"]
last15_runs = feat_panel["last15_sprints"] + feat_panel["last15_hsr"]

feat["sprint_share"] = feat_panel["cumul_sprints"] / cumul_runs.replace(0, np.nan)
feat["runs_per_minute_played"] = cumul_runs / feat_panel["mins_played_so_far"]
feat["top_third_run_share"] = feat_panel["cumul_n_top_runs"] / cumul_runs.replace(0, np.nan)
feat["peak_minus_mean_speed"] = feat_panel["cumul_peak_speed"] - feat_panel["cumul_mean_max_speed"]
feat["distance_per_run"] = feat_panel["cumul_distance"] / cumul_runs.replace(0, np.nan)

last15_intensity = feat_panel["last15_sprints"] / 15.0
cumul_intensity = feat_panel["cumul_sprints"] / feat_panel["mins_played_so_far"]
feat["last15_sprint_uplift"] = last15_intensity / (cumul_intensity + EPS)

last15_dist_intensity = feat_panel["last15_distance"] / 15.0
cumul_dist_intensity = feat_panel["cumul_distance"] / feat_panel["mins_played_so_far"]
feat["last15_distance_uplift"] = last15_dist_intensity / (cumul_dist_intensity + EPS)

feat["top_third_last15_share"] = feat_panel["last15_n_top_runs"] / last15_runs.replace(0, np.nan)

# Replace inf and clip ratios that explode when cumulative is near zero.
feat = feat.replace([np.inf, -np.inf], np.nan)
feat["last15_sprint_uplift"] = feat["last15_sprint_uplift"].clip(upper=10)
feat["last15_distance_uplift"] = feat["last15_distance_uplift"].clip(upper=10)
feat.describe().T[["count", "mean", "std", "min", "50%", "max"]]


,count,mean,std,min,50%,max
player_appearance_id,5227.0,43441.3300,7479.8245,39421.0000,41253.0000,65789.0000
sprint_share,5227.0,0.1801,0.1035,0.0000,0.1818,1.0000
runs_per_minute_played,5227.0,0.5913,0.2705,0.0095,0.5846,2.7500
top_third_run_share,5227.0,0.2826,0.1893,0.0000,0.2703,1.0000
peak_minus_mean_speed,5227.0,1.7504,0.8042,0.0000,1.7639,6.9865
distance_per_run,5227.0,43.7698,94.2929,3.9181,13.1849,526.0795
last15_sprint_uplift,5227.0,0.6322,0.7058,0.0000,0.5556,5.9995
last15_distance_uplift,5227.0,0.6989,0.5467,0.0000,0.7992,6.9999
top_third_last15_share,3937.0,0.2788,0.2411,0.0000,0.2500,1.0000


In [21]:
"""Merge engineered features onto main and bin against scored_after."""
m_aug = main.merge(feat, on=["player_appearance_id", "checkpoint"], how="left")

ENG_BIN_SPECS = {
    "sprint_share":           [-1, 0.0, 0.25, 0.5, 1.01],
    "runs_per_minute_played": [-1, 0.0, 0.5, 1.0, 100],
    "top_third_run_share":    [-1, 0.0, 0.25, 0.5, 1.01],
    "peak_minus_mean_speed":  [-1, 0.0, 0.5, 1.5, 100],
    "distance_per_run":       [-1, 0.0, 8.0, 12.0, 1000],
    "last15_sprint_uplift":   [-1, 0.0, 0.5, 1.5, 11],
    "last15_distance_uplift": [-1, 0.0, 0.5, 1.5, 11],
    "top_third_last15_share": [-1, 0.0, 0.5, 1.0, 1.01],
}

eng_rows = []
for f, edges in ENG_BIN_SPECS.items():
    t = binned_rate(m_aug, f, "scored_after", bins=edges)
    t["feature"] = f
    eng_rows.append(t)
eng_df = pd.concat(eng_rows, ignore_index=True)[["feature", "bin", "n", "n_pos", "rate", "ci_lo", "ci_hi"]]
eng_df


,feature,bin,n,n_pos,rate,ci_lo,ci_hi
0,sprint_share,"(-1.001, 0.0]",352,20,0.0568,0.0371,0.0861
1,sprint_share,"(0.0, 0.25]",2439,142,0.0582,0.0496,0.0682
2,sprint_share,"(0.25, 0.5]",554,40,0.0722,0.0535,0.0968
3,sprint_share,"(0.5, 1.01]",18,1,0.0556,0.0099,0.2576
4,sprint_share,NaN,123,0,0.0000,0.0000,0.0303
5,runs_per_minute_played,"(-1.001, 0.0]",0,0,NaN,NaN,NaN
6,runs_per_minute_played,"(0.0, 0.5]",1302,68,0.0522,0.0414,0.0657
7,runs_per_minute_played,"(0.5, 1.0]",1883,117,0.0621,0.0521,0.0740
8,runs_per_minute_played,"(1.0, 100.0]",178,18,0.1011,0.0649,0.1542
9,runs_per_minute_played,NaN,123,0,0.0000,0.0000,0.0303


### Section D - findings

Read the binned-rate table above. Per-feature verdict logic:

* **Keep** if any non-baseline bin has a CI that does not overlap the
  pooled baseline (~5.8%) AND the trend is monotone or has a clear peak.
* **Cautious keep** if the trend is monotone but CIs overlap baseline -
  worth retaining for tree-model split signal.
* **Drop** if all bins overlap the baseline.

Interactions to consider for Section F: each engineered feature x
`position`, since position confounded the cumul shot signal in shots EDA.


## Section E - Redundancy + significance

Two passes:

1. **Pearson matrix** on the union of (existing main run columns) + (kept
   engineered candidates from D). Pairs with `|rho| >= 0.95` are
   collinear by construction; the rule from shots EDA is to keep the
   simpler / parent variable.
2. **Cluster-robust GLM** of `scored_after` on each candidate, clustering
   standard errors on `fixture_id`, with Benjamini-Hochberg FDR adjustment
   across the full feature set. This honours within-fixture correlation
   (multiple players per match) and the multiple-testing burden.


In [22]:
"""Pearson correlation matrix."""
RUN_MAIN = list(RUN_BIN_SPECS.keys())
ENG_FEATS = list(ENG_BIN_SPECS.keys())
ALL_FEATS = RUN_MAIN + ENG_FEATS

corr = m_aug[ALL_FEATS].corr(method="pearson")
hi = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .rename("pearson")
        .reset_index()
        .rename(columns={"level_0": "a", "level_1": "b"})
)
hi["abs"] = hi["pearson"].abs()
hi.sort_values("abs", ascending=False).head(15)


,a,b,pearson,abs
58,last15_mean_max_speed,last15_peak_speed,0.9566,0.9566
175,cumul_peak_speed,peak_minus_mean_speed,0.9177,0.9177
153,cumul_mean_max_speed,cumul_peak_speed,0.9050,0.9050
50,last15_distance,distance_per_run,0.8900,0.8900
43,last15_distance,cumul_distance,0.8390,0.8390
140,cumul_distance,distance_per_run,0.8197,0.8197
233,top_third_run_share,top_third_last15_share,0.8074,0.8074
96,cumul_sprints,cumul_hsr,0.8031,0.8031
29,last15_hsr,runs_per_minute_played,0.7743,0.7743
81,last15_peak_speed,cumul_peak_speed,0.6985,0.6985


In [23]:
"""Cluster-robust GLM + BH-FDR (statsmodels)."""
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


def cluster_robust_p(df: pd.DataFrame, feature: str) -> tuple[float, float, int]:
    sub = df[[feature, "scored_after", "fixture_id"]].dropna()
    if sub[feature].std() == 0 or len(sub) < 30:
        return float("nan"), float("nan"), len(sub)
    try:
        m = smf.logit(f"scored_after ~ {feature}", data=sub).fit(
            disp=False, maxiter=200,
            cov_type="cluster", cov_kwds={"groups": sub["fixture_id"]},
        )
        return float(m.params[feature]), float(m.pvalues[feature]), len(sub)
    except Exception:
        return float("nan"), float("nan"), len(sub)


rows = []
for f in ALL_FEATS:
    coef, p, n = cluster_robust_p(m_aug, f)
    rho = m_aug[[f, "scored_after"]].corr().iloc[0, 1]
    rows.append({"feature": f, "n": n, "pearson_r": rho, "coef": coef, "p_raw": p})
sig_df = pd.DataFrame(rows)

mask = sig_df["p_raw"].notna()
adj = np.full(len(sig_df), np.nan)
if mask.sum() > 0:
    _, adj_p, _, _ = multipletests(sig_df.loc[mask, "p_raw"].values, method="fdr_bh")
    adj[mask.values] = adj_p
sig_df["p_bh"] = adj
sig_df["bh_q05"] = sig_df["p_bh"] < 0.05
sig_df.sort_values("p_bh").reset_index(drop=True)


,feature,n,pearson_r,coef,p_raw,p_bh,bh_q05
0,last15_mean_max_speed,3486,0.0617,2.5367e-01,1.6592e-14,2.9866e-13,True
1,top_third_last15_share,3240,0.0821,1.3198e+00,1.5975e-08,1.4378e-07,True
2,last15_peak_speed,3486,0.0687,2.0793e-01,4.7531e-08,2.8519e-07,True
3,top_third_run_share,3363,0.1036,2.0019e+00,1.0544e-06,4.7447e-06,True
4,last15_hsr,3486,0.0534,5.8078e-02,2.9807e-05,1.0731e-04,True
5,cumul_mean_max_speed,3486,0.0382,1.9222e-01,5.4125e-05,1.6237e-04,True
6,runs_per_minute_played,3363,0.0674,9.3295e-01,1.0243e-04,2.6339e-04,True
7,cumul_peak_speed,3486,0.0370,1.0886e-01,3.0735e-02,6.9153e-02,False
8,cumul_hsr,3486,-0.0316,-1.1022e-02,6.2413e-02,1.1294e-01,False
9,last15_sprints,3486,0.0279,7.9894e-02,6.2742e-02,1.1294e-01,False


### Section E - findings

Decisions logged from the two tables above:

* Drop any pair with `|pearson| >= 0.95` keeping the parent variable
  (e.g. if `cumul_sprints` and a derived ratio are perfectly collinear).
* For features failing BH at q=0.05, demote unless they carry signal in a
  tree model (revisit during modelling - feature importance can rescue
  features whose marginal logistic link is weak).
* Bivariate-significant features whose effect direction is implausible
  (e.g. negative coefficient on sprints) flag a confounder - usually
  position - and require an interaction term rather than the raw feature.


## Section F - Feature manifest

Final keep / drop list. Verdicts are derived from C / D / E. The notebook
cell below regenerates the manifest from the in-memory `eng_df`, `pooled_df`
and `sig_df` so the table stays consistent with the data after a re-run.


In [24]:
"""Programmatic manifest construction."""

def best_non_baseline_bin(df: pd.DataFrame, feature: str) -> tuple[float, float, float, int]:
    sub = df[df["feature"] == feature]
    if sub.empty:
        return (float("nan"),) * 3 + (0,)
    sub = sub.sort_values("rate", ascending=False).iloc[0]
    return float(sub["rate"]), float(sub["ci_lo"]), float(sub["ci_hi"]), int(sub["n"])


manifest_rows = []
for f in RUN_MAIN:
    rate, lo, hi, n = best_non_baseline_bin(pooled_df, f)
    manifest_rows.append({"feature": f, "source": "main", "best_rate": rate,
                          "ci_lo": lo, "ci_hi": hi, "n": n})
for f in ENG_FEATS:
    rate, lo, hi, n = best_non_baseline_bin(eng_df, f)
    manifest_rows.append({"feature": f, "source": "engineered", "best_rate": rate,
                          "ci_lo": lo, "ci_hi": hi, "n": n})

manifest = pd.DataFrame(manifest_rows).merge(
    sig_df[["feature", "pearson_r", "p_bh", "bh_q05"]], on="feature", how="left"
)


def verdict(row) -> str:
    if pd.isna(row["best_rate"]):
        return "drop (no data)"
    if row["bh_q05"] is True or (row["ci_lo"] is not None and row["ci_lo"] > BASELINE):
        return "KEEP"
    if row["best_rate"] > BASELINE * 1.4 and row["n"] >= 30:
        return "keep (cautious)"
    return "drop (flat)"


manifest["verdict"] = manifest.apply(verdict, axis=1)
manifest.sort_values(["verdict", "p_bh"]).reset_index(drop=True)


,feature,source,best_rate,ci_lo,ci_hi,n,pearson_r,p_bh,bh_q05,verdict
0,last15_mean_max_speed,main,0.0642,0.0548,0.0751,2258,0.0617,2.9866e-13,True,KEEP
1,top_third_last15_share,engineered,0.0897,0.0668,0.1194,457,0.0821,1.4378e-07,True,KEEP
2,last15_peak_speed,main,0.0666,0.0577,0.0767,2643,0.0687,2.8519e-07,True,KEEP
3,top_third_run_share,engineered,0.1232,0.0928,0.1619,349,0.1036,4.7447e-06,True,KEEP
4,last15_hsr,main,0.0674,0.0525,0.0861,861,0.0534,1.0731e-04,True,KEEP
5,cumul_mean_max_speed,main,0.0606,0.0521,0.0704,2607,0.0382,1.6237e-04,True,KEEP
6,runs_per_minute_played,engineered,0.1011,0.0649,0.1542,178,0.0674,2.6339e-04,True,KEEP
7,cumul_hsr,main,0.0782,0.0617,0.0987,818,-0.0316,1.1294e-01,False,KEEP
8,cumul_distance,main,0.0760,0.0610,0.0943,974,-0.0120,8.0642e-01,False,KEEP
9,last15_sprint_uplift,engineered,0.0917,0.0602,0.1375,218,-0.0038,8.7636e-01,False,KEEP


### Closing notes

* The manifest table above is the deliverable of this EDA.
* Once finalised, replicate the engineering logic in `features/runs.py`
  with a single entry-point `build_run_features(runs_df, main_df) ->
  DataFrame` keyed on `(player_appearance_id, checkpoint)`, mirroring
  `features/shots.py`.
* The same windowing rule, orphan filter, and ratio-NaN policy
  (`fillna(0)` plus a `has_run_history` indicator) carry over from shots.
